# Astro Engineering - Validation Notebook

This notebook validates our calculations against known reference values and the case study data.

## Case Study
- **Date**: December 5, 1986
- **Time**: 08:03:00 local
- **Location**: Santiago, Chile (-33.4489°, -70.6693°)
- **Timezone**: America/Santiago (UTC-3 on that date, summer time)

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

from datetime import datetime
from core.time_engine import local_to_ut, TimeEngine
from core.ephemeris import get_all_positions, get_position, EphemerisEngine
from core.houses import calculate_houses, compare_house_systems
from core.aspects import find_all_aspects, calculate_aspect_summary
from data.symbols import longitude_to_zodiacal

import pandas as pd
pd.set_option('display.max_columns', None)

## 1. Time Conversion Validation

In [ ]:
# Case study data
birth_date = datetime(1986, 12, 5, 8, 3, 0)
timezone = "America/Santiago"
latitude = -33.4489
longitude = -70.6693

# Convert time
time_result = local_to_ut(birth_date, timezone, latitude, longitude)

print("TIME CONVERSION VALIDATION")
print("=" * 50)
print(f"Local time:      {time_result.local}")
print(f"UTC:             {time_result.utc}")
print(f"Julian Day (UT): {time_result.jd_ut:.6f}")
print(f"Julian Day (TT): {time_result.jd_tt:.6f}")
print(f"Delta-T:         {time_result.delta_t:.2f} seconds")
print(f"TZ Offset:       {time_result.timezone_offset_hours} hours")
print(f"DST Active:      {time_result.dst_active}")

# Validation checks
print("\nVALIDATION CHECKS:")
print(f"- UTC should be 11:03 UTC: {time_result.utc.hour == 11 and time_result.utc.minute == 3}")
print(f"- TZ offset should be -3h: {time_result.timezone_offset_hours == -3.0}")
print(f"- JD around 2447074.96: {2447074.9 < time_result.jd_ut < 2447075.0}")

## 2. Planetary Positions Validation

In [ ]:
# Get planetary positions
positions = get_all_positions(time_result.jd_tt)

print("PLANETARY POSITIONS")
print("=" * 70)

# Display as table
data = []
for body, pos in positions.items():
    data.append({
        'Body': body.capitalize(),
        'Longitude': f"{pos.longitude_decimal:.4f}°",
        'Zodiacal': longitude_to_zodiacal(pos.longitude_decimal, False),
        'Latitude': f"{pos.latitude_decimal:.2f}°",
        'Speed': f"{pos.speed_longitude:.4f}°/d",
        'Retro': 'R' if pos.is_retrograde else '',
        'Method': pos.calculation_flag,
    })

df = pd.DataFrame(data)
print(df.to_string(index=False))

In [ ]:
# Expected values validation
print("\nEXPECTED VALUES VALIDATION")
print("=" * 50)

# Sun should be around 13° Sagittarius (243°)
sun_lon = positions['sun'].longitude_decimal
print(f"Sun: {longitude_to_zodiacal(sun_lon, False)}")
print(f"  Expected: ~13° Sagittarius (243°)")
print(f"  Actual: {sun_lon:.2f}°")
print(f"  Valid: {240 < sun_lon < 250}")

# Moon should be around 6° Aquarius (306°)
moon_lon = positions['moon'].longitude_decimal
print(f"\nMoon: {longitude_to_zodiacal(moon_lon, False)}")
print(f"  Expected: ~6° Aquarius (306°)")
print(f"  Actual: {moon_lon:.2f}°")
print(f"  Valid: {300 < moon_lon < 315}")

## 3. House Calculation Validation

In [ ]:
# Calculate houses with Placidus
houses = calculate_houses(time_result.jd_ut, latitude, longitude, 'P')

print("HOUSE CUSPS (Placidus)")
print("=" * 50)
print(f"Ascendant: {longitude_to_zodiacal(houses.ascendant, False)} ({houses.ascendant:.2f}°)")
print(f"MC:        {longitude_to_zodiacal(houses.mc, False)} ({houses.mc:.2f}°)")
print(f"Vertex:    {longitude_to_zodiacal(houses.vertex, False)} ({houses.vertex:.2f}°)")

print("\nHouse Cusps:")
for i, cusp in enumerate(houses.cusps):
    print(f"  House {i+1:2d}: {longitude_to_zodiacal(cusp, False)}")

In [ ]:
# Ascendant validation
print("\nASCENDANT VALIDATION")
print("=" * 50)
print(f"Expected: ~11° Capricorn (281°)")
print(f"Actual: {houses.ascendant:.2f}° = {longitude_to_zodiacal(houses.ascendant, False)}")
print(f"Valid: {275 < houses.ascendant < 290}")

print("\nMC VALIDATION")
print(f"Expected: ~25° Libra (205°)")
print(f"Actual: {houses.mc:.2f}° = {longitude_to_zodiacal(houses.mc, False)}")
print(f"Valid: {200 < houses.mc < 215}")

In [ ]:
# Compare house systems
print("\nHOUSE SYSTEM COMPARISON")
print("=" * 70)

comparison = compare_house_systems(
    time_result.jd_ut, latitude, longitude, 
    ['P', 'K', 'E', 'W', 'R', 'C']
)
print(comparison.to_string())

## 4. Aspect Validation

In [ ]:
# Add angles to positions for aspect calculation
positions_with_angles = dict(positions)
positions_with_angles['ascendant'] = {'longitude_decimal': houses.ascendant}
positions_with_angles['mc'] = {'longitude_decimal': houses.mc}

# Find aspects
aspects = find_all_aspects(positions_with_angles, time_result.jd_tt)

print("ASPECTS FOUND")
print("=" * 70)

# Sort by orb
sorted_aspects = sorted(aspects, key=lambda a: a.orb)

for asp in sorted_aspects[:15]:  # Top 15 tightest aspects
    print(f"{asp.body1.capitalize():12} {asp.aspect_name:13} {asp.body2.capitalize():12} "
          f"orb: {asp.orb:.2f}° {'(applying)' if asp.is_applying else '(separating)'}")

In [ ]:
# Aspect summary
summary = calculate_aspect_summary(aspects)

print("\nASPECT SUMMARY")
print("=" * 50)
print(f"Total aspects: {summary['total']}")
print(f"Major aspects: {summary['major']}")
print(f"Minor aspects: {summary['minor']}")
print(f"Applying: {summary['applying']}")
print(f"Separating: {summary['separating']}")
print(f"Average orb: {summary['average_orb']:.2f}°")

print("\nBy type:")
for asp_type, count in summary['by_type'].items():
    print(f"  {asp_type}: {count}")

## 5. Internal Consistency Checks

In [ ]:
from core.validation import run_internal_checks

# Run internal consistency checks
checks = run_internal_checks(time_result.utc)

print("INTERNAL CONSISTENCY CHECKS")
print("=" * 50)

for check_name, check_data in checks.items():
    status = "PASS" if check_data['ok'] else "FAIL"
    print(f"{check_name}:")
    print(f"  Value: {check_data['value']:.4f}")
    if 'expected_range' in check_data:
        print(f"  Expected: {check_data['expected_range']}")
    print(f"  Status: {status}")
    print()

## 6. Calculation Method Information

In [ ]:
# Display calculation method info
ephemeris = EphemerisEngine()

print("CALCULATION METHOD")
print("=" * 50)
print(f"Method: {ephemeris.calculation_method}")
print(f"Ephemeris path: {ephemeris.ephemeris_path or 'Using internal MOSEPH'}")

if ephemeris.calculation_method == 'MOSEPH':
    print("\nNote: Using Moshier algorithm (MOSEPH).")
    print("Precision: ~0.1 arcseconds for planets")
    print("For higher precision (~0.001 arcsec), download Swiss Ephemeris .se1 files.")
else:
    print("\nUsing Swiss Ephemeris data files (SWIEPH).")
    print("Precision: ~0.001 arcseconds for planets")

## 7. Summary

In [ ]:
print("VALIDATION SUMMARY")
print("=" * 70)
print()
print("Case Study: 1986-12-05 08:03:00 Santiago, Chile")
print()
print("Time Conversion:")
print(f"  UTC: {time_result.utc.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  JD (UT): {time_result.jd_ut:.6f}")
print(f"  Delta-T: {time_result.delta_t:.1f}s")
print()
print("Key Positions:")
print(f"  Sun: {longitude_to_zodiacal(positions['sun'].longitude_decimal, False)}")
print(f"  Moon: {longitude_to_zodiacal(positions['moon'].longitude_decimal, False)}")
print(f"  Ascendant: {longitude_to_zodiacal(houses.ascendant, False)}")
print(f"  MC: {longitude_to_zodiacal(houses.mc, False)}")
print()
print(f"Aspects found: {len(aspects)}")
print(f"Calculation method: {ephemeris.calculation_method}")
print()
print("All validations completed successfully!")